# 🎤 Chatbot Vocal - Qwen 2.5 7B + Whisper Large v3

Chatbot vocal utilisant:
- **STT**: Faster Whisper Large v3
- **LLM**: Qwen 2.5 7B Instruct
- **TTS**: Edge TTS
- **Interface**: Gradio

In [ ]:
# Installation des dépendances
!pip install torch transformers faster-whisper gradio edge-tts pydub accelerate bitsandbytes -q

# Vérification GPU
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Imports et configuration
import os
import tempfile
import numpy as np
import asyncio
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from faster_whisper import WhisperModel
from faster_whisper.utils import download_model
import edge_tts

# Configuration pour Colab
torch.cuda.empty_cache()

# Configuration des modèles
WHISPER_MODEL = "large-v3"
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
WHISPER_DIR = "/content/whisper_cache"

# Variables globales
whisper_model = None
qwen_tokenizer = None
qwen_model = None
conversation_history = []

def check_whisper_exists():
    return os.path.exists(f'{WHISPER_DIR}/large-v3') and len(os.listdir(f'{WHISPER_DIR}/large-v3')) > 3

def download_models():
    """Téléchargement intelligent des modèles"""
    # Whisper seulement
    if check_whisper_exists():
        print('✅ Whisper déjà présent en session, réutilisation...')
    else:
        print('⬇️ Téléchargement Whisper large-v3 (3GB)...')
        try:
            os.makedirs(WHISPER_DIR, exist_ok=True)
            download_model('large-v3', output_dir=WHISPER_DIR)
            print('✅ Whisper téléchargé')
        except Exception as e:
            print(f'❌ Erreur Whisper : {e}')
    
    print('🎯 Modèles prêts !')

def load_models():
    """Charger les modèles une seule fois"""
    global whisper_model, qwen_tokenizer, qwen_model

    if whisper_model is None:
        print("🔄 Chargement Whisper Large v3...")
        whisper_model = WhisperModel(
            f"{WHISPER_DIR}/large-v3",
            device="cuda" if torch.cuda.is_available() else "cpu",
            compute_type="float16" if torch.cuda.is_available() else "int8"
        )
        print("✅ Whisper chargé")

    if qwen_tokenizer is None or qwen_model is None:
        print("🔄 Chargement Qwen 2.5 7B (4-bit)...")
        print("⬇️ Téléchargement Qwen2.5-7B-Instruct (7GB)...")

        # Configuration 4-bit pour économiser la mémoire
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
        qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL,
            quantization_config=bnb_config if torch.cuda.is_available() else None,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True
        )
        print("✅ Qwen chargé")

def transcribe_audio(audio_file) -> str:
    """STT avec Faster Whisper Large v3"""
    load_models()

    try:
        if audio_file is None:
            return ""

        segments, _ = whisper_model.transcribe(
            audio_file,
            language="fr",
            vad_filter=True,
            beam_size=5
        )

        transcript = ""
        for segment in segments:
            transcript += segment.text.strip() + " "

        return transcript.strip()

    except Exception as e:
        print(f"Erreur STT: {e}")
        return ""

def generate_response(text: str) -> str:
    """LLM avec Qwen 2.5 7B"""
    global conversation_history

    if not text:
        return ""

    try:
        messages = [
            {"role": "system", "content": "Tu es un assistant vocal français. Réponds de manière concise et naturelle."}
        ]
        messages.extend(conversation_history[-6:])
        messages.append({"role": "user", "content": text})

        prompt = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = qwen_tokenizer(prompt, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = inputs.to("cuda")

        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.7,
                do_sample=True,
                pad_token_id=qwen_tokenizer.eos_token_id
            )

        response = qwen_tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()

        conversation_history.extend([
            {"role": "user", "content": text},
            {"role": "assistant", "content": response}
        ])

        if len(conversation_history) > 12:
            conversation_history = conversation_history[-12:]

        return response

    except Exception as e:
        print(f"Erreur LLM: {e}")
        return "Je rencontre un problème technique."

async def synthesize_speech(text: str) -> str:
    """TTS avec Edge TTS"""
    try:
        temp_file = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        temp_path = temp_file.name
        temp_file.close()

        communicate = edge_tts.Communicate(text, "fr-FR-DeniseNeural")
        await communicate.save(temp_path)

        return temp_path if os.path.exists(temp_path) else None

    except Exception as e:
        print(f"Erreur TTS: {e}")
        return None

def process_audio(audio_file):
    """Pipeline principal: Audio → Texte → Réponse → Audio"""
    if audio_file is None:
        return None, "Aucun audio détecté"

    # STT
    transcript = transcribe_audio(audio_file)
    if not transcript:
        return None, "Transcription échouée"

    print(f"👂 Utilisateur: '{transcript}'")

    # LLM
    response = generate_response(transcript)
    if not response:
        return None, transcript

    print(f"💬 Assistant: '{response}'")

    # TTS
    try:
        audio_path = asyncio.run(synthesize_speech(response))
        return audio_path, f"**Vous:** {transcript}\n\n**Assistant:** {response}"
    except Exception as e:
        print(f"Erreur synthèse: {e}")
        return None, f"**Vous:** {transcript}\n\n**Assistant:** {response}"

def create_interface():
    """Interface Gradio pour Colab"""

    with gr.Blocks(title="Chatbot Vocal - Qwen 2.5 7B", theme=gr.themes.Soft()) as interface:
        gr.Markdown("# 🎤 Chatbot Vocal avec Qwen 2.5 7B")
        gr.Markdown("**STT:** Faster Whisper Large v3 | **LLM:** Qwen 2.5 7B (4-bit) | **TTS:** Edge TTS")

        with gr.Row():
            with gr.Column():
                audio_input = gr.Audio(
                    sources=["microphone"],
                    type="filepath",
                    label="🎙️ Enregistrez votre message"
                )

                submit_btn = gr.Button("💬 Traiter", variant="primary")
                clear_btn = gr.Button("🗑️ Effacer historique", variant="secondary")

            with gr.Column():
                audio_output = gr.Audio(
                    label="🔊 Réponse audio",
                    autoplay=True
                )

                text_output = gr.Markdown(
                    label="📝 Conversation",
                    value="Prêt à discuter !"
                )

        # Actions
        submit_btn.click(
            fn=process_audio,
            inputs=[audio_input],
            outputs=[audio_output, text_output]
        )

        clear_btn.click(
            fn=lambda: (None, "Historique effacé !"),
            outputs=[audio_output, text_output]
        ).then(
            fn=lambda: conversation_history.clear()
        )

    return interface

# Téléchargement des modèles
print("🚀 Initialisation du chatbot vocal...")
download_models()

# Pré-chargement des modèles
load_models()

print("✅ Modèles prêts !")
print("🌐 Lancement de l'interface Gradio...")

# Interface Gradio
interface = create_interface()

# Lancement public pour Colab
interface.launch(
    share=True,
    server_name="0.0.0.0",
    server_port=7860,
    show_error=True
)